# CS231 - Hieuthudam
**Tên:** …  
**Phương pháp:** …

Toàn bộ code đọc dữ liệu, khung đặc trưng, huấn luyện và đánh giá nằm trong notebook này.
Sao chép thành tên riêng trong thư mục CS231, chọn kernel `.venv/bin/python`, chạy từ trên xuống.
Bạn tự viết vào mục 2 và 3. Khi đổi phương pháp, dùng Restart Kernel rồi Run All để tránh dùng biến cũ.

## 1. Đọc dữ liệu

Mỗi bảng có ba cột `image_path`, `class_name`, `label`. Đường dẫn đầy đủ của ảnh là `DATA_DIR / image_path`.
Nhãn số 0..22 theo thứ tự `categories.txt`. Giữ nguyên Split 1 cho mọi thành viên.

Chạy notebook từ thư mục CS231. Nếu dữ liệu nằm chỗ khác, sửa `DATA_DIR` bên dưới.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data/minc-2500")


def load_dataset():
    class_names = (DATA_DIR / "categories.txt").read_text().splitlines()
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    assert len(class_to_idx) == 23, "Cần đủ 23 lớp."
    tables = []
    for filename, per_class in [("train1.txt", 2125), ("validate1.txt", 125), ("test1.txt", 250)]:
        paths = (DATA_DIR / "labels" / filename).read_text().splitlines()
        names = [Path(path).parent.name for path in paths]
        df = pd.DataFrame({"image_path": paths, "class_name": names})
        df["label"] = df["class_name"].map(class_to_idx)
        assert df["label"].notna().all(), f"Nhãn không hợp lệ: {filename}"
        assert len(df) == 23 * per_class and df["class_name"].nunique() == 23
        assert df.groupby("class_name").size().eq(per_class).all()
        assert not df["image_path"].duplicated().any()
        assert all((DATA_DIR / path).is_file() for path in paths), "Có ảnh không tồn tại."
        tables.append(df)
    train, val, test = [set(df["image_path"]) for df in tables]
    assert train.isdisjoint(val) and train.isdisjoint(test) and val.isdisjoint(test)
    return tables[0], tables[1], tables[2], class_names


train_df, val_df, test_df, class_names = load_dataset()
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()
y_pred_val = None
y_pred_test = None

print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
train_df.head()

## 2. Trích xuất đặc trưng — tự viết

Tạo `X_train`, `X_val` theo đúng thứ tự hàng trong các bảng. `X_train[i]` phải tương ứng `y_train[i]`.

Nếu dùng scaler/PCA hoặc bước cần học tham số: chỉ fit trên train, rồi transform validation/test. Ghi lại tham số bạn sử dụng. Chưa cần xử lý test khi đang thử cấu hình.

In [ ]:
# Viết code trích xuất đặc trưng tại đây.
# X_train = ...
# X_val = ...

## 3. Huấn luyện và dự đoán validation — tự viết

Huấn luyện bằng train. Dự đoán validation thành `y_pred_val`: một nhãn số nguyên 0..22 cho mỗi ảnh, giữ đúng thứ tự `val_df`.

In [ ]:
# Viết code huấn luyện tại đây.
# model = ...
# model.fit(X_train, y_train)
# y_pred_val = model.predict(X_val)

## 4. Đánh giá validation

Hàm `evaluate` bên dưới dùng chung cho mọi phương pháp, không cần tự viết lại. Nhận nhãn dự đoán là số nguyên 0..22, cùng thứ tự ảnh.

Chọn cấu hình bằng **Macro-F1 trên validation**; xem thêm Accuracy, Precision/Recall/F1 từng lớp và confusion matrix.
Macro-F1 lấy trung bình F1 trên đủ 23 lớp; trường hợp chia cho 0 tính điểm 0. Điểm ở thang 0..1.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from pandas.api.types import is_integer_dtype
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


def evaluate(y_true, y_pred, class_names, title="Validation"):
    """Nhận nhãn số theo đúng thứ tự ảnh; điểm nằm trong 0..1."""
    labels = list(range(len(class_names)))
    truth, prediction = pd.Series(y_true), pd.Series(y_pred)
    if len(truth) == 0 or len(truth) != len(prediction):
        raise ValueError("Cần cùng số nhãn thật và nhãn dự đoán, không rỗng.")
    for values in (truth, prediction):
        if not is_integer_dtype(values.dtype) or not values.isin(labels).all():
            raise ValueError("Nhãn phải là số nguyên theo thứ tự trong categories.txt.")
    report = classification_report(
        truth, prediction, labels=labels, target_names=class_names,
        output_dict=True, zero_division=0,
    )
    metrics = {
        "accuracy": float(accuracy_score(truth, prediction)),
        "macro_f1": report["macro avg"]["f1-score"],
    }
    print(title)
    display(pd.DataFrame([metrics]))
    display(pd.DataFrame(report).T.loc[class_names])
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(
        truth, prediction, labels=labels, display_labels=class_names,
        normalize="true", include_values=False, xticks_rotation=90,
        cmap="Blues", ax=ax,
    )
    ax.set_title(f"{title} — confusion matrix (tỷ lệ theo từng hàng)")
    plt.tight_layout()
    plt.show()
    return metrics


In [ ]:
if y_pred_val is not None:
    val_metrics = evaluate(y_val, y_pred_val, class_names, title="Validation")
else:
    print("Viết phần đặc trưng và huấn luyện ở trên để có y_pred_val.")

## 5. Test sau khi chốt phương pháp

Sau khi chọn cấu hình bằng validation, tự viết phần tạo `X_test` và `y_pred_test` bên dưới. Dùng cùng mô hình, cách trích xuất và trạng thái chuẩn hóa đã học trên train. Không fit lại trên test.

So sánh các phương pháp đã chốt bằng cùng test. Không chỉnh tham số dựa trên điểm test. Trong template này, mô hình được học trên train, chưa gộp train và validation.

In [ ]:
# Chỉ điền khi đã chốt phương pháp.
# X_test = ...
# y_pred_test = model.predict(X_test)

In [ ]:
if y_pred_test is not None:
    test_metrics = evaluate(y_test, y_pred_test, class_names, title="Test")
else:
    print("Chưa đánh giá test.")

## 6. Ghi kết quả

| Phương pháp và tham số | Validation Macro-F1 | Test Accuracy | Test Macro-F1 |
|---|---|---|---|
| … | … | … | … |

- Những lớp dễ nhầm và ví dụ ảnh:
- Nhận xét về ưu điểm, hạn chế:

Chỉ điền điểm test sau khi chốt cấu hình. Bảng test chỉ so sánh những phương pháp đánh giá trên cùng danh sách ảnh.

Tài liệu công thức đánh giá: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html